In [1]:
print("Google Gen AI notebook loaded")

Google Gen AI notebook loaded


In [2]:
# Imports
import sys
import pathlib

# Add the project's root directory to the Python path
sys.path.append(pathlib.Path("../").resolve().as_posix())

# Configurations
seed = 42

# Paths
DATA_DIR = pathlib.Path("../data/")
ENC2017_ROOT = DATA_DIR / "enc2017"
UD_ET_EDT_ROOT = DATA_DIR / "ud_et_edt"
HOMONYMS_ROOT = DATA_DIR / "homonymous_word_forms"

ENC2017_DIRS = {
    "processed": ENC2017_ROOT / "processed",
    "raw": ENC2017_ROOT / "raw",
}

UD_ET_EDT_DIRS = {
    "processed": UD_ET_EDT_ROOT / "processed",
    "raw": UD_ET_EDT_ROOT / "raw",
}

HOMONYMS_DIRS = {
    "processed": HOMONYMS_ROOT / "processed",
    "annotations": HOMONYMS_ROOT / "annotations",
}

OUTPUT_DIR = pathlib.Path("../outputs/")

MODEL_DIR = pathlib.Path("../models/")

In [3]:
import json
import os
import random
import time

import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types

In [4]:
"""
Google Gen AI setup for Estonian marked-word rewriting.

This cell configures the Gen AI client, reads the API key from the environment,
and prepares the model call for JSON output.
"""

load_dotenv(".env", verbose=True)

api_key = os.getenv("GOOGLE_API_KEY")
model = "gemini-3.1-flash-lite-preview"
client = genai.Client(api_key=api_key)

print(f"Using model: {model}")

Using model: gemini-3.1-flash-lite-preview


## Google Gen AI Prompting for Estonian Sentence Rewriting

This notebook uses Google's Gen AI SDK to rewrite Estonian sentences by replacing exactly one marked token in angle brackets (`<...>`) with a context-appropriate alternative.

The prompt asks the model to produce a JSON object with the rewritten sentence, a chosen replacement, and a short candidate list to make downstream parsing reliable.


### Testing on a single sentence


In [95]:
# System prompt: enforce Estonian rewriting behaviour and JSON output
system_prompt = """You are an Estonian sentence rewriting assistant.
Your task is to replace exactly one marked token in angle brackets <...> with a context-appropriate alternative.

Rules:
- Keep the rewritten sentence natural and grammatically correct in Estonian.
- Preserve tense, number, case, agreement, punctuation, and capitalisation.
- If the marked token is a proper name (person, place, organisation), replace it with another plausible proper name.
- Internally consider multiple alternatives, then choose the best one for the full sentence context.
- Return strict JSON only. Do not add explanations or markdown.

JSON schema:
{
  "id": string,
  "candidates": array of 10 strings,
  "chosen": string,
  "rewritten": string
}
"""

# Few-shot example to anchor the output format
few_shot_user = """Sentence ID: ex-1
Sentence: Ma näen <maja>.
"""

few_shot_assistant = json.dumps(
    {
        "id": "ex-1",
        "candidates": [
            "ehitist",
            "hoonet",
            "elamut",
            "rajatist",
            "korterit",
            "kodu",
            "eluaset",
            "hoonestust",
            "majapidamist",
            "maja",
        ],
        "chosen": "ehitist",
        "rewritten": "Ma näen ehitist.",
    },
    ensure_ascii=False,
    indent=2,
)

# Real input sentence
user_prompt = """Sentence ID: 1
Sentence: Põhja-Koreaga alustas Jaapan läbirääkimisi 1992. aastal, kuid need katkesid <aasta> hiljem just samal põhjusel, mille üle nüüdki vaieldakse.
"""

contents = [
    types.Content(role="user", parts=[types.Part.from_text(text=few_shot_user)]),
    types.Content(role="model", parts=[types.Part.from_text(text=few_shot_assistant)]),
    types.Content(role="user", parts=[types.Part.from_text(text=user_prompt)]),
]

In [96]:
# Request a JSON response from Gemini / Google Gen AI
response = client.models.generate_content(
    model=model,
    contents=contents,
    config=types.GenerateContentConfig(
        system_instruction=system_prompt,
        # temperature=0.3,
        top_p=0.9,
        max_output_tokens=1000,
        response_mime_type="application/json",
    ),
)

In [97]:
content = response.text
result = json.loads(content)

print("Raw JSON response:")
print(json.dumps(result, ensure_ascii=False, indent=2))
print("\nRewritten sentence:")
print(result["rewritten"])

Raw JSON response:
{
  "id": "1",
  "candidates": [
    "kuu",
    "päeva",
    "nädala",
    "tunni",
    "minuti",
    "hetke",
    "ajal",
    "pooleteise",
    "kolme",
    "viie"
  ],
  "chosen": "kuu",
  "rewritten": "Põhja-Koreaga alustas Jaapan läbirääkimisi 1992. aastal, kuid need katkesid kuu hiljem just samal põhjusel, mille üle nüüdki vaieldakse."
}

Rewritten sentence:
Põhja-Koreaga alustas Jaapan läbirääkimisi 1992. aastal, kuid need katkesid kuu hiljem just samal põhjusel, mille üle nüüdki vaieldakse.


### Testing on a sample of 10 sentences with batch processing and error handling


In [7]:
# Load the homonyms dataset to get the sentences we want to rewrite with Gen AI
homonyms_df = pd.read_parquet(
    HOMONYMS_DIRS["processed"] / "homonyms_overall_sentences.parquet"
)
display(homonyms_df.head())

,num,inflection_type,sentence,word,word_span,label,source
0,1,1,Edinburghi agulite mehe Irvine Welshi ja Glasg...,võitja,"[74, 80]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-0...
1,1,1,"Normi-aktiveerimise teooria (Schwartz, 1970) o...",teooria,"[20, 27]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-0...
2,1,1,"""Ehk oleks mõttekas ka mõni selleteemaline hoi...",kampaania,"[51, 60]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-0...
3,1,1,"""Minu otsus oli õige ning teeksin kõik sama mo...",õige,"[16, 20]",[sg n],infl_type_01_1000_v1_project-1-at-2024-11-23-0...
4,1,1,Itaalia president ütles Venemaa riigipea auks ...,Itaalia,"[0, 7]",[sg g],infl_type_01_1000_v1_project-1-at-2024-11-23-0...


In [8]:
# System prompt: enforce Estonian rewriting behaviour and JSON output
system_prompt = """You are an Estonian sentence rewriting assistant.
Your task is to replace exactly one marked token in angle brackets <...> with a context-appropriate alternative.

Rules:
- Keep the rewritten sentence natural and grammatically correct in Estonian.
- Preserve tense, number, case, agreement, punctuation, and capitalisation.
- If the marked token is a proper name (person, place, organisation), replace it with another plausible proper name.
- Internally consider multiple alternatives, then choose the best one for the full sentence context.
- Return strict JSON only. Do not add explanations or markdown.

JSON schema:
{
  "id": string,
  "candidates": array of 10 strings,
  "chosen": string,
  "rewritten": string
}
"""

# Few-shot example to anchor the output format
few_shot_user = """Sentence ID: ex-1
Sentence: Ma näen <maja>.
"""

few_shot_assistant = json.dumps(
    {
        "id": "ex-1",
        "candidates": [
            "ehitist",
            "hoonet",
            "elamut",
            "rajatist",
            "korterit",
            "kodu",
            "eluaset",
            "hoonestust",
            "majapidamist",
            "maja",
        ],
        "chosen": "ehitist",
        "rewritten": "Ma näen ehitist.",
    },
    ensure_ascii=False,
    indent=2,
)

# Real input sentence
user_prompt = """Sentence ID: 1
Sentence: Põhja-Koreaga alustas Jaapan läbirääkimisi 1992. aastal, kuid need katkesid <aasta> hiljem just samal põhjusel, mille üle nüüdki vaieldakse.
"""

contents = [
    types.Content(role="user", parts=[types.Part.from_text(text=few_shot_user)]),
    types.Content(role="model", parts=[types.Part.from_text(text=few_shot_assistant)]),
    types.Content(role="user", parts=[types.Part.from_text(text=user_prompt)]),
]


# Create a user prompt template for the real sentences, using the same format as the few-shot example
def create_user_prompt(sentence_id, sentence, word_span):
    # Mark the target word with angle brackets
    start, end = word_span
    marked_sentence = (
        sentence[:start] + "<" + sentence[start:end] + ">" + sentence[end:]
    )
    return f"""Sentence ID: {sentence_id}
Sentence: {marked_sentence}
"""


def append_result_to_json(file_path, record):
    """Append one record to a JSON array stored in file_path."""
    file_path.parent.mkdir(parents=True, exist_ok=True)
    # If the file exists and is non-empty, load the existing data; otherwise start with an empty list
    if file_path.exists() and file_path.stat().st_size > 0:
        with open(file_path, "r", encoding="utf-8") as f:
            try:
                data = json.load(f)
            except json.JSONDecodeError:
                data = []
    else:
        data = []

    if not isinstance(data, list):
        data = []

    data.append(record)
    # Write the updated list back to the file with pretty formatting
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


def rewrite_sentence_with_genai(sentence_id, sentence, word_span, word, label):
    """Rewrite one sentence by replacing the marked target word via Gemini."""
    user_prompt = create_user_prompt(
        sentence_id=sentence_id, sentence=sentence, word_span=word_span
    )
    # The few-shot example is included in every call to ensure the model understands the expected JSON format and the task.
    local_contents = [
        types.Content(role="user", parts=[types.Part.from_text(text=few_shot_user)]),
        types.Content(
            role="model", parts=[types.Part.from_text(text=few_shot_assistant)]
        ),
        types.Content(role="user", parts=[types.Part.from_text(text=user_prompt)]),
    ]
    # Request a JSON response from Gemini / Google Gen AI
    response = client.models.generate_content(
        model=model,
        contents=local_contents,
        config=types.GenerateContentConfig(
            system_instruction=system_prompt,
            # temperature=0.3,
            top_p=0.9,
            max_output_tokens=1000,
            response_mime_type="application/json",
        ),
    )
    # Parse the JSON response and enrich it with metadata for later analysis
    content = response.text or "{}"
    parsed = json.loads(content)

    # Calculate the new word span in the rewritten sentence, if possible
    rewritten = parsed.get("rewritten", "")
    chosen = parsed.get("chosen", "")
    if rewritten and chosen:
        new_start = rewritten.find(chosen)
        if new_start != -1:
            new_end = new_start + len(chosen)
            parsed["new_word_span"] = [new_start, new_end]
        else:
            parsed["new_word_span"] = None
    else:
        parsed["new_word_span"] = None

    # Keep useful metadata for later analysis/debugging.
    parsed["source_sentence"] = sentence
    parsed["target_word"] = word
    parsed["word_span"] = word_span.astype(
        int
    ).tolist()  # Convert numpy array to list for JSON serialization
    parsed["label"] = label

    return parsed


def get_latest_processed_id(output_file):
    """Get the highest sentence ID that has already been processed and saved in the output file."""
    if output_file.exists() and output_file.stat().st_size > 0:
        with open(output_file, "r", encoding="utf-8") as f:
            data = json.load(f)
            if isinstance(data, list) and len(data) > 0:
                return max(int(record["id"]) for record in data if "id" in record)
    return -1  # No valid records found, start from the beginning


def is_transient_error(exc: Exception) -> bool:
    """Heuristically detect retryable errors such as rate limit and network hiccups."""
    message = str(exc).lower()
    transient_markers = [
        "429",
        "rate",
        "quota",
        "too many requests",
        "timeout",
        "timed out",
        "connection",
        "temporar",
        "unavailable",
        "503",
        "502",
        "504",
        "internal",
    ]
    return any(marker in message for marker in transient_markers)


def rewrite_with_retry(sentence_id, sentence, word_span, word, label, config):
    """Call rewrite function with exponential backoff on transient errors."""
    for attempt in range(1, config["max_attempts"] + 1):
        try:
            return rewrite_sentence_with_genai(
                sentence_id=sentence_id,
                sentence=sentence,
                word_span=word_span,
                word=word,
                label=label,
            )
        except Exception as exc:
            should_retry = is_transient_error(exc) and attempt < config["max_attempts"]
            if not should_retry:
                raise

            backoff = min(
                config["max_backoff_seconds"],
                config["initial_backoff_seconds"] * (2 ** (attempt - 1)),
            )
            sleep_s = backoff + random.uniform(0.0, config["jitter_seconds"])
            print(
                f"Retry {attempt}/{config['max_attempts'] - 1} for sentence_id={sentence_id} "
                f"after transient error: {exc}. Sleeping {sleep_s:.2f}s..."
            )
            time.sleep(sleep_s)

In [9]:
example_sentence = homonyms_df.iloc[0]["sentence"]
example_span = homonyms_df.iloc[0]["word_span"]
example_id = 0
example_prompt = create_user_prompt(example_id, example_sentence, example_span)
print("Example user prompt for a real sentence:")
print(example_prompt)

Example user prompt for a real sentence:
Sentence ID: 0
Sentence: Edinburghi agulite mehe Irvine Welshi ja Glasgow tööliskirjaniku, Bookeri <võitja> James Kelmani puhul võib tõlketõrke tekitada keelekasutus - inglise inglise keelele demonstratiivselt vastanduv proletaarne Scots.



In [10]:
output_file = OUTPUT_DIR / "genai_rewrites_sample.json"  # Output file for results
max_rows = 1  # Change or remove this limit for larger runs

# gemini-3.1-flash-lite-preview has strict request limits, so we throttle and retry safely.
config = {
    # Pacing configuration to avoid hitting rate limits
    "base_delay_seconds": 4.0,
    "jitter_seconds": 1.0,
    # Retry configuration for transient failures
    "max_attempts": 5,
    "initial_backoff_seconds": 2.0,
    "max_backoff_seconds": 60.0,
}

# Start from the next unprocessed sentence based on the output file contents
latest_id = get_latest_processed_id(output_file)
unprocessed_df = homonyms_df[homonyms_df.index > latest_id]
print(
    f"Starting processing from sentence ID > {latest_id + 1}. Total unprocessed sentences: {len(unprocessed_df)}"
)
print("Next sentence to process:")
display(unprocessed_df.head(1))

Starting processing from sentence ID > 99. Total unprocessed sentences: 7787
Next sentence to process:


,num,inflection_type,sentence,word,word_span,label,source
99,1,1,""" Tehtud sai registrite, reformide, aruandluse...",kalmistu,"[209, 217]",[sg g],infl_type_01_1000_v1_project-1-at-2024-11-23-0...


In [ ]:
# Batch rewrite loop: call Gen AI and append each result to a JSON file
processed = 0
for sentence_id, (sentence, word_span, word, label) in enumerate(
    zip(
        unprocessed_df["sentence"],
        unprocessed_df["word_span"],
        unprocessed_df["word"],
        unprocessed_df["label"],
    ),
    start=latest_id + 1,
):
    if sentence_id >= max_rows:
        break

    try:
        result = rewrite_with_retry(
            sentence_id=sentence_id,
            sentence=sentence,
            word_span=word_span,
            word=word,
            label=label,
            config=config,
        )
        append_result_to_json(output_file, result)
        print(f"[{processed + 1}] Saved sentence_id={sentence_id}")
    except Exception as exc:
        error_record = {
            "id": str(sentence_id),
            "source_sentence": sentence,
            "target_word": word,
            "word_span": word_span.astype(int).tolist(),
            "error": str(exc),
        }
        append_result_to_json(output_file, error_record)
        print(f"[{processed + 1}] Error on sentence_id={sentence_id}: {exc}")

    processed += 1

    # Additional pacing between successful/failed items to avoid bursty traffic.
    if sentence_id + 1 < max_rows:
        sleep_s = config["base_delay_seconds"] + random.uniform(
            0.0, config["jitter_seconds"]
        )
        print(f"Sleeping {sleep_s:.2f}s before next request...")
        time.sleep(sleep_s)

print(f"Finished. Processed {processed} rows. Results appended to: {output_file}")

[1] Saved sentence_id=0
Finished. Processed 1 rows. Results appended to: ..\outputs\genai_rewrites_sample.json


In [46]:
homonyms_df.iloc[8]

num                                                                1
inflection_type                                                    1
sentence           Siiditee John W. Orrison, endine CSX Transport...
word                                                       piiramatu
word_span                                                 [165, 174]
label                                                         [sg g]
source             infl_type_01_1000_v1_project-1-at-2024-11-23-0...
Name: 8, dtype: object

In [11]:
# Open the output file
if output_file.exists():
    with open(output_file, "r", encoding="utf-8") as f:
        data = json.load(f)
        # Iterate over the homonym dataset and add the form label and new word span as metadata to the corresponding result record
        for id, record in enumerate(data):
            matching_row = homonyms_df[homonyms_df.index == id]
            if not matching_row.empty:
                form_label = matching_row.iloc[0]["label"]
                record["form_label"] = form_label.tolist()
                record["id"] = str(id)  # Ensure the ID is a string for consistency
                # Calculate the new word span in the rewritten sentence, if possible
                rewritten = record.get("rewritten", "")
                chosen = record.get("chosen", "")
                if rewritten and chosen:
                    new_start = rewritten.find(chosen)
                    if new_start != -1:
                        new_end = new_start + len(chosen)
                        record["new_word_span"] = [new_start, new_end]
                    else:
                        record["new_word_span"] = None
                else:
                    record["new_word_span"] = None
            else:
                print(f"No matching row found in homonyms_df for record ID: {id}")
    # Save the enriched results back to the file
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

In [ ]:
# Open LLM sample annotation data
llm_samples_df = pd.read_json(OUTPUT_DIR / "genai_rewrites_sample.json", orient="records")
cols = llm_samples_df.columns.tolist()
cols.insert(3, cols.pop(cols.index("new_word_span"))) # Move new_word_span to after the third column
cols.insert(4, cols.pop(cols.index("form_label"))) # Move form_label to after new_word_span
llm_samples_df = llm_samples_df[cols]
# Save back to JSON
llm_samples_df.to_json(OUTPUT_DIR / "genai_rewrites_sample.json", orient="records", indent=2, force_ascii=False)